----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Content layer - Custom Instruments - Historical Data
This standalone notebook demonstrates historical events, intraday and interday summaries, multi-instrument requests and asynchronous retrieval with `lseg.data.content.custom_instruments`.

**Prerequisites:** a configured LSEG Data session with access to Custom Instruments, historical data and the underlying RICs. Creating the examples requires `trapi.synthetic.crud` for platform sessions. See [Manage and Search](EX-2.12.01-CustomInstruments-ManageAndSearch.ipynb) for instrument management and the current basket/UDC interface limitations.

The notebook creates two unique formula instruments on the service. Run the cleanup cell before closing the session, even if a data request fails. Before re-running the creation cell, delete the previous instruments.

#### Learn more

Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials about the LSEG Data Library for Python.

#### Getting Help and Support

For questions about the API, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run the notebook from its own directory. The configuration file in the project's `Configuration` directory supplies the session settings and credentials.

In [ ]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with

In [ ]:
import asyncio
from datetime import timedelta
from uuid import uuid4

import lseg.data as ld
from lseg.data.content import custom_instruments as ci

## Open the data session
The default session is selected from `lseg-data.config.json`. Use `ld.open_session("platform.rdp")` instead to select that configured platform session explicitly.

In [ ]:
ld.open_session()

## Create the example instruments
Retain the full symbols returned by the service. Appending each symbol immediately allows cleanup of the first instrument if creation of the second fails.

In [ ]:
symbols = []
for multiplier in (2, 3):
    instrument = ci.manage.create_formula(
        symbol=f"Hist{uuid4().hex[:8]}",
        formula=f"GBP=*{multiplier}",
        instrument_name=f"GBP Times {multiplier}",
        currency="USD",
        time_zone="LON",
        description="Custom Instruments historical data example",
    )
    symbols.append(instrument.symbol)

symbols

## Retrieve recent events
`events.Definition` requests individual historical events rather than consolidated bars. `count` limits the number of returned records (1 to 10000). The active test requests 15 events.

Responses expose a DataFrame through `data.df` and the service payload through `data.raw`. Available history depends on the instrument, underlying data, entitlements and service readiness; a newly created instrument may not have data immediately. An unavailable history request may raise `LDError`. Inspect the error instead of substituting fabricated data.

In [ ]:
events_response = ci.events.Definition(
    universe=symbols[0],
    count=15,
).get_data()
events_response.data.df

In [ ]:
events_response.data.raw

### Multiple instruments and a relative time window
Both events and summaries accept a single symbol or a list. Negative `timedelta` values are relative to now; `timedelta(0)` represents now. These parameters also accept dates, datetimes and ISO date/time strings; explicit timestamps must be in UTC. A recent relative window avoids hard-coded dates that become stale.

In [ ]:
multi_events_response = ci.events.Definition(
    universe=symbols,
    start=timedelta(hours=-24),
    end=timedelta(0),
    count=100,
).get_data()
multi_events_response.data.df

## Retrieve intraday summaries
`summaries.Definition` consolidates the historical data at the requested interval. The tests exercise both `ONE_MINUTE` and `DAILY`. This example requests up to 30 one-minute summaries without restricting the field list.

In [ ]:
intraday_response = ci.summaries.Definition(
    universe=symbols[0],
    interval=ci.Intervals.ONE_MINUTE,
    count=30,
).get_data()
intraday_response.data.df

## Retrieve interday summaries for multiple instruments
Use `DAILY` for daily consolidation, or other supported intervals such as `WEEKLY`. The following request asks for up to 30 records within the last 90 days. Requesting a date range does not guarantee that history exists throughout that range.

In [ ]:
daily_response = ci.summaries.Definition(
    universe=symbols,
    interval=ci.Intervals.DAILY,
    start=timedelta(days=-90),
    end=timedelta(0),
    count=30,
).get_data()
daily_response.data.df

## Retrieve events and summaries asynchronously
`get_data_async()` is the asynchronous counterpart of `get_data()`. Independent requests can be awaited together with `asyncio.gather`. Jupyter supports top-level `await`; do not call `asyncio.run()` inside an already running notebook event loop. Request errors propagate to the cell.

In [ ]:
async_events, async_summaries = await asyncio.gather(
    ci.events.Definition(universe=symbols[0], count=15).get_data_async(),
    ci.summaries.Definition(
        universe=symbols[0],
        interval=ci.Intervals.DAILY,
        count=10,
    ).get_data_async(),
)
display(async_events.data.df)
display(async_summaries.data.df)

## Delete the example instruments
Run this cell even if a previous data request fails. Only the symbols created in this notebook are deleted. A symbol is removed from the local list after a successful deletion, so cleanup can be retried if a deletion fails.

In [ ]:
while symbols:
    ci.manage.delete(universe=symbols[-1])
    symbols.pop()

## Close the session

In [ ]:
ld.close_session()